In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

NEGATION_WORDS = {"not", "no", "never", "n't", "cannot", "cant", "without"}
DEFAULT_STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "and", "or", "but", "with",
    "this", "that", "these", "those", "it", "its", "as", "so", "very",
    "i", "my", "me", "we", "our", "you", "your",
} - NEGATION_WORDS

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> list:
    return text.split()

def remove_stop_words(tokens: list, stop_words: set = None) -> list:
    stop_words = stop_words if stop_words is not None else DEFAULT_STOP_WORDS
    return [t for t in tokens if t not in stop_words]

def preprocess(text: str, remove_stops: bool = True) -> str:
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    if remove_stops:
        tokens = remove_stop_words(tokens)
    return " ".join(tokens)

In [2]:
df = pd.read_csv('feedback.csv')
df['clean_feedback'] = df['feedback'].apply(preprocess)
df.head()

,feedback,sentiment,category,clean_feedback
0,Payment keeps failing every time I checkout,negative,payment,payment keeps failing every time checkout
1,The payment gateway failed after entering the OTP,negative,payment,payment gateway failed after entering otp
2,I was charged twice for the same order,negative,payment,charged twice same order
3,Payment went through smoothly this time,positive,payment,payment went through smoothly time
4,"Refund was processed within a day, thank you",positive,payment,refund processed within day thank


In [3]:
vectorizer_uni = TfidfVectorizer()
vectors_uni = vectorizer_uni.fit_transform(df['clean_feedback'])

print('Vocabulary size (unigrams):', len(vectorizer_uni.get_feature_names_out()))
print(list(vectorizer_uni.get_feature_names_out())[:15])

Vocabulary size (unigrams): 142
['add', 'after', 'allow', 'amazing', 'app', 'application', 'arrived', 'arriving', 'bug', 'button', 'buttons', 'can', 'cannot', 'charged', 'checkout']


In [4]:
vectorizer_bi = TfidfVectorizer(ngram_range=(1, 2))
vectors_bi = vectorizer_bi.fit_transform(df['clean_feedback'])

print('Vocabulary size (uni+bigrams):', len(vectorizer_bi.get_feature_names_out()))
print([f for f in vectorizer_bi.get_feature_names_out() if ' ' in f][:15])

Vocabulary size (uni+bigrams): 293
['add dark', 'add multi', 'after entering', 'after last', 'allow exporting', 'app crashed', 'app extremely', 'app slow', 'application freezes', 'application okay', 'application slow', 'application updated', 'arriving phone', 'bug logs', 'bug search']


In [5]:
row_index = 0
row = vectors_bi[row_index].toarray().flatten()
feature_names = vectorizer_bi.get_feature_names_out()

top_terms = sorted(zip(feature_names, row), key=lambda x: x[1], reverse=True)[:10]
print('Feedback:', df['feedback'].iloc[row_index])
for term, score in top_terms:
    if score > 0:
        print(f'{term:25s} {score:.3f}')

Feedback: Payment keeps failing every time I checkout
failing every             0.339
time checkout             0.339
checkout                  0.305
every                     0.305
every time                0.305
keeps                     0.305
keeps failing             0.305
payment keeps             0.305
failing                   0.281
time                      0.281
